# ARC26 Qwen/TRM mean-NLL validation48

Train the unchanged per-puzzle Qwen LoRA, skip DFS, and rescore the retained Qwen 8x4 candidate pool plus saved TRM attempts using mean target-token NLL.


In [ ]:
import os

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["OMP_NUM_THREADS"] = "12"


In [ ]:
MODE = "validation"

CODE_DATASET_ROOT = "/kaggle/input/datasets/yuvraj/arc2026"
MODEL_PATH = "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1"
COMP_ROOT = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"

VALIDATION_KEYS = [
    "0934a4d8",
    "135a2760",
    "136b0064",
    "13e47133",
    "142ca369",
    "16b78196",
    "16de56c4",
    "1818057f",
    "195c6913",
    "1ae2feb7",
    "20270e3b",
    "20a9e565",
    "21897d95",
    "221dfab4",
    "247ef758",
    "269e22fb",
    "271d71e2",
    "28a6681f",
    "291dc1e1",
    "2b83f449",
    "2ba387bc",
    "2c181942",
    "2d0172a1",
    "31f7f899",
    "332f06d7",
    "35ab12c3",
    "36a08778",
    "38007db0",
    "3a25b0d8",
    "3dc255db",
    "3e6067c3",
    "409aa875",
    "446ef5d2",
    "45a5af55",
    "4a21e3da",
    "4c3d4a41",
    "4c416de3",
    "4c7dc4dd",
    "4e34c42c",
    "53fb4810",
    "5545f144",
    "581f7754",
    "58490d8a",
    "58f5dbd5",
    "5961cc34",
    "5dbc8537",
    "62593bfd",
    "64efde09"
]
NPROCS = 4
PROFILE_TIMINGS = True
VALIDATION_END_TIME_HOURS = 4.0

WORK_NOTEBOOK_ROOT = "/kaggle/working/arc26_qwen_trm_mean_nll_validation48"
WORK_CODE_DIR = WORK_NOTEBOOK_ROOT + "/ARC-AGI1/qwen_baseline"
WRITABLE_UNSLOTH_PARENT = "/kaggle/working/qwen_trm_mean_nll_stack"
FIXED_CANDIDATE_DIR = "/kaggle/working/qwen_trm_fixed_candidates"
OUTPUT_DIR = "/kaggle/working/qwen_trm_mean_nll_scored"
SUMMARY_PATH = "/kaggle/working/qwen_trm_mean_nll_validation48_summary.json"
TEST_PATH = f"{COMP_ROOT}/arc-agi_evaluation_challenges.json"
SOLUTION_PATH = f"{COMP_ROOT}/arc-agi_evaluation_solutions.json"
END_TIME_HOURS = VALIDATION_END_TIME_HOURS
RUN_INFERENCE = True


In [ ]:
import os

IS_KAGGLE_RERUN = os.getenv("KAGGLE_IS_COMPETITION_RERUN", "").lower() in {
    "1", "true", "yes", "y", "on"
}
assert not IS_KAGGLE_RERUN, "This is a validation-only scoring notebook"
EFFECTIVE_MODE = "validation"
RESET_RUN_ARTIFACTS = True
SUBMISSION_PATH = SUMMARY_PATH
SELECTED_KEYS = VALIDATION_KEYS
print("test_path =", TEST_PATH)
print("output_dir =", OUTPUT_DIR)
print("fixed_candidate_dir =", FIXED_CANDIDATE_DIR)
print("selected_keys =", SELECTED_KEYS)


In [ ]:
import importlib.util
import os
import shutil
import sys
from pathlib import Path

assert Path(CODE_DATASET_ROOT).exists(), f"Missing code dataset root: {CODE_DATASET_ROOT}"
assert Path(MODEL_PATH).exists(), f"Missing model path: {MODEL_PATH}"
assert Path(TEST_PATH).exists(), f"Missing challenge path: {TEST_PATH}"
assert Path(os.environ["TRITON_PTXAS_PATH"]).exists(), os.environ["TRITON_PTXAS_PATH"]
if SOLUTION_PATH is not None:
    assert Path(SOLUTION_PATH).exists(), f"Missing solution path: {SOLUTION_PATH}"

os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"

os.chdir("/kaggle/working")
print("setup cwd =", os.getcwd())

if RESET_RUN_ARTIFACTS:
    for path in [WORK_NOTEBOOK_ROOT, OUTPUT_DIR, WRITABLE_UNSLOTH_PARENT]:
        shutil.rmtree(path, ignore_errors=True)
    try:
        Path(SUBMISSION_PATH).unlink()
    except FileNotFoundError:
        pass
    for path in Path("/kaggle/working").glob("worker_train_*"):
        if path.is_file():
            path.unlink()

for module_name in ["unsloth", "transformers", "torch"]:
    spec = importlib.util.find_spec(module_name)
    print(module_name, spec.origin if spec else "MISSING")


In [ ]:
import os
import shutil
from pathlib import Path

src = Path(CODE_DATASET_ROOT)
dst = Path(WORK_NOTEBOOK_ROOT)
shutil.copytree(src, dst)

required_files = [
    "starter.py",
    "arc_solver.py",
    "arc_rescoring.py",
    "analyze_qwen_trm_mean_nll.py",
    "qwen_trm_validation48_fixed_candidates.zip",
]
for name in required_files:
    assert Path(WORK_CODE_DIR, name).is_file(), f"arc2026 is stale: missing {name}"

starter_source = Path(WORK_CODE_DIR, "starter.py").read_text()
solver_source = Path(WORK_CODE_DIR, "arc_solver.py").read_text()
rescoring_source = Path(WORK_CODE_DIR, "arc_rescoring.py").read_text()
assert "--fixed-candidate-mean-nll" in starter_source
assert "fixed_candidate_mean_nll" in solver_source
assert "normalize_by_answer_tokens" in rescoring_source

shutil.rmtree(FIXED_CANDIDATE_DIR, ignore_errors=True)
shutil.unpack_archive(
    Path(WORK_CODE_DIR, "qwen_trm_validation48_fixed_candidates.zip"),
    FIXED_CANDIDATE_DIR,
)
assert len(list(Path(FIXED_CANDIDATE_DIR).iterdir())) == 1581
print("fixed candidate pool ready:", FIXED_CANDIDATE_DIR)


In [ ]:
spec = importlib.util.find_spec("unsloth")
assert spec is not None and spec.submodule_search_locations
mounted_unsloth = Path(next(iter(spec.submodule_search_locations)))
qwen_source = (mounted_unsloth / "models" / "qwen3.py").read_text()
assert "A = flash_attn_func(Qnn, Knn, Vnn)" in qwen_source

writable_parent = Path(WRITABLE_UNSLOTH_PARENT)
writable_unsloth = writable_parent / "unsloth"
shutil.copytree(mounted_unsloth, writable_unsloth)

sys.path.insert(0, WORK_CODE_DIR)
from patch_unsloth_qwen3_multitoken import PATCH_MARKER, patch_unsloth

changed = patch_unsloth(writable_unsloth)
assert PATCH_MARKER in (writable_unsloth / "models" / "qwen3.py").read_text()
print("writable_unsloth =", writable_unsloth)
print("patched =", [str(path) for path in changed])

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = str(writable_parent) + os.pathsep + RUN_ENV.get("PYTHONPATH", "")


In [ ]:
import json
import subprocess
import sys
import time

cmd = [
    sys.executable,
    "starter.py",
    "--test-path", TEST_PATH,
    "--model-path", MODEL_PATH,
    "--output-dir", OUTPUT_DIR,
    "--nprocs", str(NPROCS),
    "--fixed-candidate-dir", FIXED_CANDIDATE_DIR,
    "--fixed-candidate-mean-nll",
    "--eval-color-permutations", "4",
    "--end-time", str(time.time() + END_TIME_HOURS * 3600),
    "--keys-json", json.dumps(VALIDATION_KEYS),
]
if PROFILE_TIMINGS:
    cmd.append("--profile-timings")
print("running:", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=WORK_CODE_DIR, env=RUN_ENV, check=True)


In [ ]:
import json
import subprocess
import sys

cmd = [
    sys.executable,
    "analyze_qwen_trm_mean_nll.py",
    "--challenges", TEST_PATH,
    "--solutions", SOLUTION_PATH,
    "--candidate-dir", OUTPUT_DIR,
    "--keys-json", json.dumps(VALIDATION_KEYS),
    "--output", SUMMARY_PATH,
]
print("analyzing:", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=WORK_CODE_DIR, env=RUN_ENV, check=True)
print(Path(SUMMARY_PATH).read_text())


In [ ]:
import shutil
from pathlib import Path

archive_path = shutil.make_archive(
    "/kaggle/working/qwen_trm_mean_nll_scored_candidates",
    "zip",
    root_dir=OUTPUT_DIR,
)
print("scored_candidate_archive =", archive_path)
print("scored_candidate_archive_bytes =", Path(archive_path).stat().st_size)
print("summary_path =", SUMMARY_PATH)
shutil.rmtree(OUTPUT_DIR)
shutil.rmtree(FIXED_CANDIDATE_DIR)
shutil.rmtree(WORK_NOTEBOOK_ROOT)
shutil.rmtree(WRITABLE_UNSLOTH_PARENT)
print("large working directories removed")
